# Viewer for DRESDEN + FLUX

3. **For every other metric**, on the remaining images only: the `N` crops with the **lowest** values and the `N`
   with the **highest** values (numerically: for `brisque` and `niqe` lower = better). No mask contour is drawn on the crops.

4. **Combined figures** (`COMBINED = True`): ONE image with all the metrics (one row per metric) showing the `N` worst
   images, and ONE image with the `N` best. With `QUALITY_ORDER = True` the rows are put in the right place for
   `niqe` and `brisque`, where lower = better: their *worst* images are the ones with the **highest** values and
   their *best* the ones with the lowest. With `QUALITY_ORDER = False` the two images are plain numeric
   lowest / highest for every metric.

In [ ]:
from pathlib import Path
import shutil
import sys

# ---- settings ----------------------------------------------------------------------------------
SCRIPTS = Path(".")                                   # folder containing common.py
OUT = Path("/var/home/da/Downloads/results-dresden-flux/flux")       # --out folder of the scripts
SAVE_DIR = OUT / "viewer"                             # where figures and csv are saved
SAVE = True                                           # False = display only
COPY_CROPS = False                                    # True = also copy the selected crops to SAVE_DIR/<metric>/{lowest,highest}/
DPI = 150

#FILTER_METRIC = "color_dev_frac_in"
#THRESHOLD = 0.02                                       # keep values < THRESHOLD
INCLUDE_NAN = False                                   # True = also keep images without FILTER_METRIC
KIND = "all"                                          # face | mask | full | all
N = 6                                                 # images per extreme
METRICS_TO_SHOW = ["musiq", "clipiqa", "brisque", "niqe", "maniqa", "clip_score", "mps_score"]
LOWER_IS_BETTER = {"brisque", "niqe"}                 # metrics where a lower value is better

PER_METRIC = True                                     # one figure per metric (lowest + highest rows)
COMBINED = True                                       # one figure with all the metrics: N worst, and one with the N best
QUALITY_ORDER = True                                  # True: worst/best (niqe and brisque flipped: worst = highest values)
                                                      # False: plain numeric lowest/highest for every metric

sys.path.insert(0, str(SCRIPTS.resolve()))
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from common import load_merged

df = load_merged(OUT)
if KIND != "all":
    df = df[df["crop_kind"] == KIND]
print(f"{len(df)} images, crop_kind: {df['crop_kind'].value_counts().to_dict()}")
print("available metrics:", [m for m in METRICS_TO_SHOW if m in df and df[m].notna().any()])
print("missing or empty metrics:", [m for m in METRICS_TO_SHOW if m not in df or not df[m].notna().any()])

## Functions

In [ ]:
SELECTED = []          # list of the selected images, saved to extremes.csv


def record(metric, side, rank, r):
    """side = 'low' | 'high' (numeric end of the metric). quality_end tells whether that end is the worst or the best."""
    worst = (side == "low") != (metric in LOWER_IS_BETTER)
    SELECTED.append({"metric": metric, "extreme": "lowest" if side == "low" else "highest",
                     "quality_end": "worst" if worst else "best", "rank": rank, "value": float(r[metric]),
                     "file": r["file"], "crop_file": r["crop_file"]})


def copy_crops(metric, side, part):
    if SAVE and COPY_CROPS:
        dst = SAVE_DIR / metric / ("lowest" if side == "low" else "highest")
        dst.mkdir(parents=True, exist_ok=True)
        for j, (_, r) in enumerate(part.iterrows(), 1):
            shutil.copy(OUT / r["crop_file"], dst / f"{j:02d}_{r[metric]:.4g}_{Path(r['crop_file']).name}")


def load_crop(row):
    img = cv2.imread(str(OUT / row["crop_file"]), cv2.IMREAD_COLOR)
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def short_name(f):
    return Path(f).stem.split("_inpainted")[0][:28]


def save_fig(fig, name):
    if SAVE:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        p = SAVE_DIR / f"{name}.png"
        fig.savefig(p, dpi=DPI, bbox_inches="tight")
        print("saved", p)


def draw_tile(ax, row, metric):
    img = load_crop(row)
    if img is not None:
        ax.imshow(img)
    flags = row.get("flags") if isinstance(row.get("flags"), str) else ""
    title = f"{row[metric]:.4g}\n{short_name(row['file'])}"
    if flags:
        title += f"\n{flags}"
    ax.set_title(title, fontsize=7.5, color="#b00020" if flags else "black")
    ax.set_xticks([]); ax.set_yticks([])


def histogram(ax, values, marks, title, thr=None):
    ax.hist(values, bins=60, color="#9aa5b1")
    for v in marks:
        ax.axvline(v, color="#d1495b", lw=0.8, alpha=0.8)
    if thr is not None:
        ax.axvline(thr, color="black", lw=1.5, ls="--")
    ax.set_title(title, fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_yticks([])


def show_quantiles(d, metric, n=24, cols=6, tile=2.6, thr=None, name=None):
    """n images spread evenly from the lowest to the highest value of the metric."""
    d = d[d[metric].notna()].sort_values(metric, kind="mergesort").reset_index(drop=True)
    if d.empty:
        print(f"No image with {metric}")
        return
    pos = np.unique(np.round(np.linspace(0, len(d) - 1, min(n, len(d)))).astype(int))
    sel = d.iloc[pos]
    rows = int(np.ceil(len(sel) / cols))
    fig = plt.figure(figsize=(cols * tile, 1.6 + rows * (tile + 0.5)))
    gs = gridspec.GridSpec(rows + 1, cols, height_ratios=[1.6 / tile] + [1] * rows, hspace=0.45, wspace=0.05)
    histogram(fig.add_subplot(gs[0, :]), d[metric], sel[metric],
              f"{metric}  ({len(d)} images, ascending)" + (f"  -  dashed line: threshold {thr}" if thr is not None else ""), thr)
    for i, (_, r) in enumerate(sel.iterrows()):
        draw_tile(fig.add_subplot(gs[1 + i // cols, i % cols]), r, metric)
    save_fig(fig, name or f"{metric}_quantiles")
    plt.show()


def extremes(d, metric, n=N):
    """(n lowest, n highest from the top down, number of valid images). Never overlapping."""
    d = d[d[metric].notna()].sort_values(metric, kind="mergesort")
    k = min(n, len(d) // 2)
    return d.head(k), d.tail(k).iloc[::-1], len(d)


def show_extremes(d, metric, n=N, tile=2.6):
    low, high, tot = extremes(d, metric, n)
    if len(low) == 0:
        print(f"{metric}: too few valid images ({tot}), skipped")
        return
    k = len(low)
    fig = plt.figure(figsize=(max(k, 3) * tile, 1.6 + 2 * (tile + 0.6)))
    gs = gridspec.GridSpec(3, k, height_ratios=[1.6 / tile, 1, 1], hspace=0.55, wspace=0.05)
    better = "  -  lower = better" if metric in LOWER_IS_BETTER else ""
    histogram(fig.add_subplot(gs[0, :]), d[metric].dropna(), list(low[metric]) + list(high[metric]),
              f"{metric}  ({tot} filtered images){better}")
    for row_i, (label, part) in enumerate((("lowest values", low), ("highest values", high)), start=1):
        for j, (_, r) in enumerate(part.iterrows()):
            ax = fig.add_subplot(gs[row_i, j])
            draw_tile(ax, r, metric)
            if j == 0:
                ax.set_ylabel(label, fontsize=9)
            record(metric, "low" if row_i == 1 else "high", j + 1, r)
    save_fig(fig, f"{metric}_extremes")
    plt.show()
    copy_crops(metric, "low", low)
    copy_crops(metric, "high", high)


def side_of(metric, which):
    """Numeric end ('low' or 'high') shown in the combined figure `which` ('first' = worst/lowest, 'second' = best/highest).
    With QUALITY_ORDER the metrics where lower is better are flipped, so each figure holds the same kind of images."""
    flip = QUALITY_ORDER and metric in LOWER_IS_BETTER
    if which == "first":
        return "high" if flip else "low"
    return "low" if flip else "high"


def show_combined(d, which, n=N, tile=2.6, title_h=0.6):
    """One figure with one row per metric: the n worst (which='first') or n best (which='second') images."""
    names = ("worst", "best") if QUALITY_ORDER else ("lowest", "highest")
    name = names[0] if which == "first" else names[1]
    parts = []
    for m in METRICS_TO_SHOW:
        if m not in d or not d[m].notna().any():
            continue
        low, high, tot = extremes(d, m, n)
        if len(low) == 0:
            continue
        side = side_of(m, which)
        parts.append((m, side, low if side == "low" else high))
    if not parts:
        print("no metric available for the combined figure")
        return

    # layout in inches: square tiles that fill their cells, title_h of room above each tile
    cols, rows, wspace = max(len(p[2]) for p in parts), len(parts), 0.05
    fig_w = max(cols, 3) * tile
    left_in, right_in, bottom_in = 0.45, 0.05, 0.05
    cell = (fig_w - left_in - right_in) / (cols + (cols - 1) * wspace)
    height = rows * (cell + title_h) + bottom_in
    fig = plt.figure(figsize=(fig_w, height))
    gs = gridspec.GridSpec(rows, cols, hspace=title_h / cell, wspace=wspace,
                           left=left_in / fig_w, right=1 - right_in / fig_w,
                           top=1 - title_h / height, bottom=bottom_in / height)
    for i, (m, side, part) in enumerate(parts):
        for j, (_, r) in enumerate(part.iterrows()):
            ax = fig.add_subplot(gs[i, j])
            draw_tile(ax, r, m)
            if j == 0:
                ax.set_ylabel(f"{m} {'↓' if m in LOWER_IS_BETTER else '↑'}", fontsize=9)
            record(m, side, j + 1, r)
        copy_crops(m, side, part)
    save_fig(fig, f"all_metrics_{name}")
    plt.show()

## 3. The N lowest and N highest images for each metric (one figure per metric)

In [ ]:
if PER_METRIC:
    for m in METRICS_TO_SHOW:
        if m not in df or not df[m].notna().any():
            print(f"{m}: missing or empty, skipped")
            continue
        show_extremes(df, m, N)

## 3b. All metrics in one image: the N worst, and the N best

In [ ]:
if COMBINED:
    show_combined(df, "first")      # worst (or lowest values if QUALITY_ORDER = False)

In [ ]:
if COMBINED:
    show_combined(df, "second")     # best (or highest values if QUALITY_ORDER = False)